# 🚀 TraderPath TFT Model Training (Standalone)

Self-contained notebook - repo clone gerektirmez.

**Runtime > Change runtime type > T4 GPU** seçin!

In [ ]:
# 1. GPU Kontrolü
!nvidia-smi

In [ ]:
# 2. Dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118
!pip install -q lightning>=2.0.0 pytorch-forecasting>=1.1.0
!pip install -q pandas numpy httpx loguru pyarrow nest_asyncio

In [ ]:
# 3. Imports
import asyncio
import gc
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
from pathlib import Path
from datetime import datetime, timedelta
import httpx
import json

print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Configuration
class TradeType(Enum):
    SCALP = "scalp"
    SWING = "swing"
    POSITION = "position"

@dataclass
class TradeTypeConfig:
    trade_type: TradeType
    prediction_horizons: List[int]
    min_encoder_length: int
    max_encoder_length: int
    min_prediction_length: int
    max_prediction_length: int
    min_training_samples: int
    data_interval: str
    lookback_days: int
    hidden_size: int = 64
    attention_heads: int = 4
    dropout: float = 0.1
    learning_rate: float = 0.001
    early_stopping_patience: int = 10

# Trade Type Configs
CONFIGS = {
    TradeType.SCALP: TradeTypeConfig(
        trade_type=TradeType.SCALP,
        prediction_horizons=[1, 2, 4],
        min_encoder_length=24,
        max_encoder_length=72,
        min_prediction_length=1,
        max_prediction_length=4,
        min_training_samples=15000,
        data_interval="15m",
        lookback_days=180,
        hidden_size=64,
        early_stopping_patience=8,
    ),
    TradeType.SWING: TradeTypeConfig(
        trade_type=TradeType.SWING,
        prediction_horizons=[24, 48],
        min_encoder_length=72,
        max_encoder_length=168,
        min_prediction_length=24,
        max_prediction_length=48,
        min_training_samples=15000,
        data_interval="1h",
        lookback_days=730,
        hidden_size=128,
        early_stopping_patience=10,
    ),
    TradeType.POSITION: TradeTypeConfig(
        trade_type=TradeType.POSITION,
        prediction_horizons=[168, 336],
        min_encoder_length=168,
        max_encoder_length=252,
        min_prediction_length=42,
        max_prediction_length=168,
        min_training_samples=5000,
        data_interval="4h",
        lookback_days=1095,
        hidden_size=128,
        early_stopping_patience=15,
    ),
}

print("✅ Config loaded")

In [ ]:
# 5. Data Fetcher
INTERVAL_MS = {
    "1m": 60_000, "5m": 300_000, "15m": 900_000,
    "1h": 3_600_000, "4h": 14_400_000, "1d": 86_400_000
}

async def fetch_binance_data(symbol: str, interval: str, days: int) -> pd.DataFrame:
    """Binance'den OHLCV verisi çek"""
    print(f"  Fetching {symbol}...")
    
    end_time = int(datetime.utcnow().timestamp() * 1000)
    start_time = int((datetime.utcnow() - timedelta(days=days)).timestamp() * 1000)
    
    all_klines = []
    current_start = start_time
    
    async with httpx.AsyncClient(timeout=60.0) as client:
        while current_start < end_time:
            params = {
                "symbol": f"{symbol}USDT",
                "interval": interval,
                "startTime": current_start,
                "endTime": end_time,
                "limit": 1000
            }
            
            resp = await client.get("https://api.binance.com/api/v3/klines", params=params)
            klines = resp.json()
            
            if not klines:
                break
                
            all_klines.extend(klines)
            current_start = klines[-1][6] + 1
            
            await asyncio.sleep(0.1)
    
    if not all_klines:
        return pd.DataFrame()
    
    df = pd.DataFrame(all_klines, columns=[
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_volume', 'trades', 'taker_buy_base',
        'taker_buy_quote', 'ignore'
    ])
    
    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms')
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    
    df['symbol'] = symbol
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume', 'symbol']]
    
    print(f"  ✅ {symbol}: {len(df)} candles")
    return df

print("✅ Data fetcher ready")

In [ ]:
# 6. Feature Engineering
def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Teknik indikatörler ekle"""
    df = df.copy()
    
    # Returns
    df['returns'] = df['close'].pct_change()
    df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
    
    # Moving Averages
    for period in [7, 14, 21, 50]:
        df[f'sma_{period}'] = df['close'].rolling(period).mean()
        df[f'ema_{period}'] = df['close'].ewm(span=period).mean()
    
    # RSI
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # MACD
    ema12 = df['close'].ewm(span=12).mean()
    ema26 = df['close'].ewm(span=26).mean()
    df['macd'] = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    
    # Bollinger Bands
    df['bb_mid'] = df['close'].rolling(20).mean()
    bb_std = df['close'].rolling(20).std()
    df['bb_upper'] = df['bb_mid'] + 2 * bb_std
    df['bb_lower'] = df['bb_mid'] - 2 * bb_std
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_mid']
    
    # ATR
    high_low = df['high'] - df['low']
    high_close = abs(df['high'] - df['close'].shift())
    low_close = abs(df['low'] - df['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['atr'] = tr.rolling(14).mean()
    
    # Volume features
    df['volume_sma'] = df['volume'].rolling(20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma']
    
    # Time features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    
    # Drop NaN
    df = df.dropna().reset_index(drop=True)
    
    return df

print("✅ Feature engineering ready")

In [ ]:
# ========================================
# 🔧 AYARLAR - BURADAN DEĞİŞTİRİN
# ========================================

TRADE_TYPE = TradeType.SWING  # SCALP, SWING, veya POSITION
SYMBOLS = ["BTC", "ETH"]      # Eğitilecek coinler
MAX_EPOCHS = 50               # Epoch sayısı
BATCH_SIZE = 64               # Batch size

config = CONFIGS[TRADE_TYPE]
print(f"\n📊 Trade Type: {TRADE_TYPE.value}")
print(f"📊 Symbols: {SYMBOLS}")
print(f"📊 Interval: {config.data_interval}")
print(f"📊 Lookback: {config.lookback_days} days")
print(f"📊 Encoder: {config.min_encoder_length}-{config.max_encoder_length}")
print(f"📊 Prediction: {config.min_prediction_length}-{config.max_prediction_length}")

In [ ]:
# 7. Veri Çek
print("\n📥 Step 1: Fetching data from Binance...")

async def fetch_all_data():
    all_data = []
    for symbol in SYMBOLS:
        df = await fetch_binance_data(symbol, config.data_interval, config.lookback_days)
        if len(df) > 0:
            df = add_technical_indicators(df)
            df['group'] = symbol
            df['time_idx'] = range(len(df))
            all_data.append(df)
    return pd.concat(all_data, ignore_index=True)

# Colab'da async çalıştır
import nest_asyncio
nest_asyncio.apply()

data = asyncio.get_event_loop().run_until_complete(fetch_all_data())
print(f"\n✅ Total: {len(data):,} samples, {len(data.columns)} features")

In [ ]:
# 8. Dataset Hazırla
print("\n📦 Step 2: Preparing dataset...")

# Train/Val split
train_cutoff = int(len(data) * 0.85)
train_df = data.iloc[:train_cutoff]
val_df = data.iloc[train_cutoff:]

print(f"  Train: {len(train_df):,} samples")
print(f"  Val: {len(val_df):,} samples")

# Feature columns
feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns', 'log_returns',
                'sma_7', 'sma_14', 'sma_21', 'sma_50', 'ema_7', 'ema_14', 'ema_21', 'ema_50',
                'rsi', 'macd', 'macd_signal', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_width',
                'atr', 'volume_sma', 'volume_ratio']

# TimeSeriesDataSet
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx='time_idx',
    target='close',
    group_ids=['group'],
    min_encoder_length=config.min_encoder_length,
    max_encoder_length=config.max_encoder_length,
    min_prediction_length=config.min_prediction_length,
    max_prediction_length=config.max_prediction_length,
    time_varying_known_reals=['hour', 'day_of_week'],
    time_varying_unknown_reals=[c for c in feature_cols if c in train_df.columns],
    target_normalizer=GroupNormalizer(groups=['group'], transformation='softplus'),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, val_df, predict=True, stop_randomization=True
)

train_loader = training_dataset.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=2)
val_loader = validation_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=2)

print(f"✅ Dataset ready")

In [ ]:
# 9. Model Oluştur
print("\n🧠 Step 3: Creating TFT model...")

model = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=config.learning_rate,
    hidden_size=config.hidden_size,
    attention_head_size=config.attention_heads,
    dropout=config.dropout,
    hidden_continuous_size=config.hidden_size // 2,
    lstm_layers=2,
    output_size=7,
    loss=QuantileLoss(quantiles=[0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]),
    reduce_on_plateau_patience=4,
    logging_metrics=[MAE()],
)

print(f"✅ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 10. Training
print("\n🏋️ Step 4: Training...")

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath="checkpoints",
    filename=f"tft_{TRADE_TYPE.value}_{{epoch}}_{{val_loss:.4f}}",
    monitor='val_loss',
    mode='min',
    save_top_k=3
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    gradient_clip_val=0.1,
    enable_progress_bar=True,
    callbacks=[
        pl.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=config.early_stopping_patience,
            mode='min'
        ),
        checkpoint_callback,
    ],
)

# Clear memory
gc.collect()
torch.cuda.empty_cache()

# Train!
trainer.fit(model, train_loader, val_loader)

print(f"\n✅ Training complete!")
print(f"✅ Best model: {checkpoint_callback.best_model_path}")

In [ ]:
# 11. Model Kaydet
print("\n💾 Step 5: Saving model...")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
model_filename = f"tft_{TRADE_TYPE.value}_{timestamp}.pt"

# Best model yükle
best_model = TemporalFusionTransformer.load_from_checkpoint(checkpoint_callback.best_model_path)

# Kaydet
torch.save({
    'model_state_dict': best_model.state_dict(),
    'trade_type': TRADE_TYPE.value,
    'symbols': SYMBOLS,
    'config': {
        'hidden_size': config.hidden_size,
        'attention_heads': config.attention_heads,
        'max_encoder_length': config.max_encoder_length,
        'max_prediction_length': config.max_prediction_length,
    }
}, model_filename)

print(f"✅ Model saved: {model_filename}")

In [ ]:
# 12. Google Drive'a Kaydet (Opsiyonel)
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(model_filename, f'/content/drive/MyDrive/{model_filename}')
print(f"✅ Model copied to Google Drive: {model_filename}")

In [ ]:
# 13. Dosyayı İndir
from google.colab import files
files.download(model_filename)